# Day 9 v1 — Silver Transformation: Charging Sessions (Blob CSV → Silver Delta)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions/` (Delta)

This is the **learning version** — one entity (charging_sessions), every step written explicitly.
Same teaching pattern as Day 8 v1 but for Blob CSV data instead of API JSON.

**Blob CSV Bronze structure:**
```
bronze-volume/realtime/charging_sessions/YYYY/MM/DD/HH/sessions_YYYYMMDD_HHMM.csv
```

**Real CSV columns (from actual Bronze data):**
```
session_id, charger_id, vehicle_id, station_id, customer_id,
plug_in_ts, charge_end_ts, energy_kwh, session_status, tariff_id
```

> **Note:** `updated_at` is not a CSV column — it is derived from the file path
> (year/month/day/hour extracted from `_metadata.file_path`).

**Steps:**
1. Read all Bronze CSV files with `recursiveFileLookup=true`
2. Extract `updated_at` from file path metadata
3. Cast columns to correct types
4. Trim whitespace from string columns
5. Add Silver audit columns
6. Deduplicate on `session_id` (latest `updated_at` wins)
7. Write to Silver as Delta table (full overwrite)
8. Verify the output

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions"
SILVER_PATH = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver path : {SILVER_PATH}")
print(f"Run time    : {RUN_TS}")

In [ ]:
# Cell 3 — Read all Bronze CSV files
# Unity Catalog requires _metadata.file_path (input_file_name() is blocked).
# List CSV columns explicitly in .select() alongside _metadata.file_path —
# do NOT use "*" wildcard which causes column count mismatch and value shifting.
from pyspark.sql.types import StructType, StructField, StringType

SCHEMA = StructType([
    StructField("session_id",     StringType(), True),
    StructField("charger_id",     StringType(), True),
    StructField("vehicle_id",     StringType(), True),
    StructField("station_id",     StringType(), True),
    StructField("customer_id",    StringType(), True),
    StructField("plug_in_ts",     StringType(), True),
    StructField("charge_end_ts",  StringType(), True),
    StructField("energy_kwh",     StringType(), True),
    StructField("session_status", StringType(), True),
    StructField("tariff_id",      StringType(), True),
])

CSV_COLS = [f.name for f in SCHEMA.fields]

raw_df = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .schema(SCHEMA)
    .csv(BRONZE_PATH)
    .select(
        *[F.col(c) for c in CSV_COLS],
        F.col("_metadata.file_path").alias("_file_path")
    )
)

print(f"Row count: {raw_df.count()}")
raw_df.show(3, truncate=True)

In [ ]:
# Cell 4 — Derive updated_at from file path partition (YYYY/MM/DD/HH)

raw_df = (
    raw_df
    .withColumn("_year",  F.regexp_extract(F.col("_file_path"), r"/(\d{4})/", 1))
    .withColumn("_month", F.regexp_extract(F.col("_file_path"), r"/\d{4}/(\d{2})/", 1))
    .withColumn("_day",   F.regexp_extract(F.col("_file_path"), r"/\d{4}/\d{2}/(\d{2})/", 1))
    .withColumn("_hour",  F.regexp_extract(F.col("_file_path"), r"/\d{4}/\d{2}/\d{2}/(\d{2})/", 1))
    .withColumn(
        "updated_at",
        F.to_timestamp(
            F.concat_ws(" ",
                F.concat_ws("-", F.col("_year"), F.col("_month"), F.col("_day")),
                F.col("_hour")
            ),
            "yyyy-MM-dd HH"
        )
    )
    .drop("_file_path", "_year", "_month", "_day", "_hour")
)

print("After adding updated_at:")
raw_df.select("session_id", "plug_in_ts", "energy_kwh", "updated_at").show(3, truncate=True)

In [ ]:
# Cell 5 — Cast columns to correct types
# Only select the columns that actually exist in Bronze CSV.
# Real columns: session_id, charger_id, vehicle_id, station_id, customer_id,
#               plug_in_ts, charge_end_ts, energy_kwh, session_status, tariff_id, updated_at

typed_df = raw_df.select(
    F.col("session_id").cast("string"),
    F.col("charger_id").cast("string"),
    F.col("vehicle_id").cast("string"),
    F.col("station_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("plug_in_ts").cast("timestamp"),
    F.col("charge_end_ts").cast("timestamp"),
    F.col("energy_kwh").cast("decimal(10,4)"),
    F.col("session_status").cast("string"),
    F.col("tariff_id").cast("string"),
    F.col("updated_at").cast("timestamp"),
)

print("After type casting:")
typed_df.printSchema()

In [ ]:
# Cell 6 — Trim whitespace from all string columns
# CSV files can have trailing spaces in string fields.

string_cols = [c for c, t in typed_df.dtypes if t == "string"]
trimmed_df  = typed_df
for c in string_cols:
    trimmed_df = trimmed_df.withColumn(c, F.trim(F.col(c)))

print(f"Trimmed string columns: {string_cols}")

In [ ]:
# Cell 7 — Add Silver audit columns

audited_df = (
    trimmed_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit("full"))
    .withColumn("silver_pipeline",    F.lit("pl_silver_blob_charging_sessions_v1"))
)

print("After adding audit columns:")
audited_df.printSchema()

In [ ]:
# Cell 8 — Deduplicate on session_id
# Bronze may contain the same session_id across multiple hourly CSV files
# (e.g. an in-progress session updated across two hours).
# Keep the row with the most recent updated_at per session_id.

window = Window.partitionBy("session_id").orderBy(F.col("updated_at").desc())

deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

before = audited_df.count()
after  = deduped_df.count()
print(f"Before dedup       : {before}")
print(f"After dedup        : {after}")
print(f"Duplicates removed : {before - after}")

In [ ]:
# Cell 9 — Write to Silver as Delta table (full overwrite)

(
    deduped_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print(f"Written to  : {SILVER_PATH}")
print(f"Rows written: {deduped_df.count()}")

In [ ]:
# Cell 10 — Verify Silver output

silver_df = spark.read.format("delta").load(SILVER_PATH)

print("Silver charging_sessions schema:")
silver_df.printSchema()
print(f"\nTotal rows: {silver_df.count()}")
silver_df.show(5, truncate=True)

print("\nNull check on session_id (should be 0):")
print(silver_df.filter(F.col("session_id").isNull()).count())

print("\nStatus distribution:")
silver_df.groupBy("session_status").count().orderBy("count", ascending=False).show()